# Stock Sentiment Analyzer
### A Real-Time NLP Pipeline for Financial Market Intelligence

> **Can the way people talk about a stock online predict its price movement?**  
> This project attempts to answer that question using real data, AI, and financial analysis.

## Objective

The goal of this project is to build an **end-to-end sentiment analysis pipeline** that:

1. Automatically collects real financial news headlines about any stock
2. Uses **FinBERT** (an AI model trained specifically on financial text) to score each headline as positive, negative, or neutral
3. Fetches **live stock price data** from Yahoo Finance
4. Analyzes whether public sentiment correlates with actual price movements
5. Presents everything in a clean, visual, and reproducible report

## Why This Matters

Every day, thousands of news articles and social media posts are published about publicly traded companies.  
Institutional investors spend millions on tools that analyze this data automatically.  
This project builds a simplified but **real and functional** version of that system, using open-source tools and free APIs.


## Tech Stack

| Tool | Purpose |
|---|---|
| `FinBERT` (HuggingFace) | AI model for financial sentiment analysis |
| `yfinance` | Fetching live and historical stock prices |
| `NewsAPI` | Collecting real financial news headlines |
| `Plotly` | Interactive visualizations |
| `Pandas` | Data manipulation and analysis |
| `Streamlit` | Live dashboard (separate module) |


## Project Structure

```
stock-sentiment-analyzer/
├── src/
│   ├── scraper.py      # Collects news headlines
│   ├── sentiment.py    # Scores sentiment with FinBERT
│   ├── stocks.py       # Fetches stock price data
│   └── pipeline.py     # Connects everything together
├── dashboard/
│   └── app.py          # Live Streamlit dashboard
├── notebook.ipynb      # This report
└── README.md
```



## Step 1  Setup & Imports

Before running the analysis, we load all required modules.  
Each module was built as part of this project and handles a specific part of the pipeline.

In [12]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), 'src'))

from scraper import get_news
from sentiment import analyze_sentiment, analyze_multiple
from stocks import get_stock_price, get_stock_history
from pipeline import run_analysis

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

print("✅ All modules loaded successfully!")

✅ All modules loaded successfully!




## Step 2 Data Collection

In this step we fetch **real financial news headlines** about our chosen stock using NewsAPI.

NewsAPI gives us access to thousands of news sources including Reuters, Bloomberg, CNBC, and more.

The `scraper.py` module handles this automatically, we just pass it a stock ticker and it returns a list of headlines ready for analysis.

In [13]:
# Define the stock we want to analyze
TICKER = "NVDA"
COMPANY = "NVIDIA"

print(f"Fetching headlines for {COMPANY} ({TICKER})...\n")

# Fetch headlines
headlines = get_news(TICKER, COMPANY)

# Display
for i, headline in enumerate(headlines, 1):
    print(f"{i:02d}. {headline[:100]}...")

print(f"\n✅ Total headlines collected: {len(headlines)}")

Fetching headlines for NVIDIA (NVDA)...

01. Google is on track for its worst day ever — with $500 billion in market value at stake. Alphabet (GO...
02. Alphabet heads for a $400 billion day that only Nvidia has ever topped. is on track to erase more th...
03. Nvidia, SpaceX Lead India's Growing U.S. Stock Craze. This article first appeared on GuruFocus.
Nvid...
04. Morning Minute: Robinhood Misses, But Crypto Beats and Robinhood Chain Soars. Vlad Tenev delivered t...
05. Nvidia, AMD Get Their Jollies Hooking Up With Meta's Muse. AI chipmakers Nvidia (NVDA) and Advanced ...
06. PrimeXBT Expands Into Tokenized Stocks, Bringing Tesla, SpaceX, and NVIDIA to Crypto Futures. PrimeX...
07. Meta's nearly $100 billion stock sale is a turning point for how Big Tech pays for AI. For a decade,...
08. PrimeXBT Brings Tesla, SpaceX, NVIDIA and More to Crypto Futures with New Tokenised Stocks. Castries...
09. ETFs Are Net Sellers of NVIDIA (NVDA) on Oct. 23. This article first appeared on GuruFocus.


## Step 3 Sentiment Analysis

Now we feed every headline into **FinBERT**. BERT-based AI model fine-tuned specifically on financial text.

Unlike general sentiment models, FinBERT understands financial language. For example:
- *"Revenue beats expectations"* → it knows this is **positive**
- *"Faces headwinds amid macro uncertainty"* → it knows this is **negative**

Each headline gets:
- A **label**: positive, negative, or neutral
- A **confidence score**: how sure the model is (0 to 1)
- A **sentiment score**: from -1 (very negative) to +1 (very positive)

In [14]:
print("Running FinBERT sentiment analysis on all headlines...\n")

# Analyze all headlines
sentiment_results = analyze_multiple(headlines)

# Build a dataframe
rows = []
for item in sentiment_results["results"]:
    emoji = "🟢" if item["label"] == "positive" else "🔴" if item["label"] == "negative" else "🟡"
    rows.append({
        "Sentiment": f"{emoji} {item['label'].upper()}",
        "Score": item["sentiment_score"],
        "Confidence": item["confidence"],
        "Headline": item["text"][:100] + "..." if len(item["text"]) > 100 else item["text"]
    })

df_sentiment = pd.DataFrame(rows)

print(f"✅ Analysis complete!\n")
print(f"Average Sentiment Score : {sentiment_results['average_score']:+.4f}")
print(f"Overall Sentiment Label : {sentiment_results['overall_label'].upper()}")
print(f"\nBreakdown:")
print(df_sentiment["Sentiment"].value_counts().to_string())

Running FinBERT sentiment analysis on all headlines...

✅ Analysis complete!

Average Sentiment Score : +0.1020
Overall Sentiment Label : POSITIVE

Breakdown:
Sentiment
🟡 NEUTRAL     9
🟢 POSITIVE    7
🔴 NEGATIVE    4



## Step 4  Visualizing Sentiment Results

Now that every headline has been scored, let's visualize the results.

We'll look at:
- A **bar chart** of all headline scores so we can see the distribution
- A **pie chart** showing the breakdown of positive, negative, and neutral headlines

In [22]:
# Bar Chart: Individual Headline Scores

scores = df_sentiment["Score"].tolist()
headlines_short = [f"#{i+1}" for i in range(len(scores))]
colors = ["#00c853" if s > 0.1 else "#ff4b4b" if s < -0.1 else "#ffd700" for s in scores]

fig_bar = go.Figure()

fig_bar.add_trace(go.Bar(
    x=headlines_short,
    y=scores,
    marker=dict(
        color=colors,
        line=dict(color="rgba(0,0,0,0.15)", width=1),
        opacity=0.9
    ),
    text=[f"{s:+.2f}" for s in scores],
    textposition="outside",
    textfont=dict(size=11, color="white"),
    hovertemplate="<b>Headline %{x}</b><br>Score: %{y:+.2f}<br><extra></extra>"
))

fig_bar.add_hline(
    y=0,
    line_dash="dash",
    line_color="rgba(255,255,255,0.4)",
    line_width=1.5
)

avg = sentiment_results["average_score"]
fig_bar.add_hline(
    y=avg,
    line_dash="dot",
    line_color="#00bcd4",
    line_width=2,
    annotation_text=f"  Average: {avg:+.2f}",
    annotation_font_color="#00bcd4",
    annotation_font_size=13
)

fig_bar.update_layout(
    title=dict(
        text=f"<b>Sentiment Score per Headline — {TICKER}</b>",
        font=dict(size=20, color="white"),
        x=0.5,
        xanchor="center"
    ),
    xaxis=dict(
        title=dict(text="Headline", font=dict(color="white")),
        tickfont=dict(color="white"),
        showgrid=False,
        zeroline=False
    ),
    yaxis=dict(
        title=dict(text="Sentiment Score", font=dict(color="white")),
        range=[-1.3, 1.3],
        tickfont=dict(color="white"),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#1a1a2e",
    height=450,
    margin=dict(t=80, b=60, l=60, r=40),
    showlegend=False
)

fig_bar.show()
fig_bar.write_image("iframe_figures/sentiment_bar.png", width=1000, height=600, scale=2)

In [27]:
# Donut Chart: Sentiment Breakdown

label_counts = df_sentiment["Sentiment"].value_counts()

# Clean up labels
clean_labels = [l.split(" ")[1].capitalize() for l in label_counts.index]
color_map = {"Positive": "#00c853", "Negative": "#ff4b4b", "Neutral": "#ffd700"}
chart_colors = [color_map.get(l, "#aaaaaa") for l in clean_labels]

total = label_counts.sum()
positive_pct = round((label_counts.get(f"🟢 POSITIVE", 0) / total) * 100, 1)

fig_donut = go.Figure()

fig_donut.add_trace(go.Pie(
    labels=clean_labels,
    values=label_counts.values.tolist(),
    hole=0.65,
    marker=dict(
        colors=chart_colors,
        line=dict(color="#1a1a2e", width=3)
    ),
    textinfo="label+percent",
    textfont=dict(size=14, color="white"),
    hovertemplate="<b>%{label}</b><br>Count: %{value}<br>%{percent}<extra></extra>",
    rotation=90
))

# Center annotation
fig_donut.add_annotation(
    text=f"<b>{sentiment_results['overall_label'].upper()}</b>",
    x=0.5, y=0.55,
    font=dict(size=18, color="white"),
    showarrow=False
)
fig_donut.add_annotation(
    text=f"<b>{sentiment_results['average_score']:+.2f}</b>",
    x=0.5, y=0.42,
    font=dict(size=28, color="#00bcd4"),
    showarrow=False
)
fig_donut.add_annotation(
    text="avg score",
    x=0.5, y=0.30,
    font=dict(size=12, color="rgba(255,255,255,0.5)"),
    showarrow=False
)

fig_donut.update_layout(
    title=dict(
        text=f"<b>Sentiment Breakdown — {TICKER}</b>",
        font=dict(size=20, color="white"),
        x=0.5,
        xanchor="center"
    ),
    paper_bgcolor="#1a1a2e",
    plot_bgcolor="#1a1a2e",
    height=450,
    legend=dict(
        font=dict(color="white", size=13),
        bgcolor="rgba(0,0,0,0)",
        orientation="v",
        x=0.85,
        y=0.5
    ),
    margin=dict(t=80, b=40, l=40, r=40)
)

fig_donut.show()
fig_donut.write_image("iframe_figures/sentiment_donut.png", width=800, height=600, scale=2)


## Step 5 Stock Price Analysis

Now let's look at the actual stock price data for **NVIDIA (NVDA)** over the past 30 days.

We fetch this data directly from **Yahoo Finance** using the `yfinance` library.

We will look at:
- The **closing price** trend over the last 30 days
- The **daily trading volume**: how much activity is happening
- **Day high vs day low**: the price range each day

In [28]:
# Fetch 30 days of historical data
history = get_stock_history(TICKER, "1mo")
history = history.reset_index()

# Candlestick Chart
fig_candle = go.Figure()

fig_candle.add_trace(go.Candlestick(
    x=history["Date"],
    open=history["Open"],
    high=history["High"],
    low=history["Low"],
    close=history["Close"],
    increasing=dict(line=dict(color="#00c853"), fillcolor="#00c853"),
    decreasing=dict(line=dict(color="#ff4b4b"), fillcolor="#ff4b4b"),
    name="Price"
))

# Add 7-day moving average
history["MA7"] = history["Close"].rolling(window=7).mean()
fig_candle.add_trace(go.Scatter(
    x=history["Date"],
    y=history["MA7"],
    line=dict(color="#00bcd4", width=2, dash="dot"),
    name="7-Day MA"
))

fig_candle.update_layout(
    title=dict(
        text=f"<b>{TICKER} Price History — Last 30 Days</b>",
        font=dict(size=20, color="white"),
        x=0.5,
        xanchor="center"
    ),
    xaxis=dict(
        title=dict(text="Date", font=dict(color="white")),
        tickfont=dict(color="white"),
        showgrid=False,
        zeroline=False,
        rangeslider=dict(visible=False)
    ),
    yaxis=dict(
        title=dict(text="Price (USD)", font=dict(color="white")),
        tickfont=dict(color="white"),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#1a1a2e",
    height=450,
    margin=dict(t=80, b=60, l=60, r=40),
    legend=dict(
        font=dict(color="white"),
        bgcolor="rgba(0,0,0,0)"
    )
)

fig_candle.show()
fig_candle.write_image("iframe_figures/price_candlestick.png", width=1000, height=600, scale=2)

In [29]:
# Volume Bar Chart
fig_vol = go.Figure()

fig_vol.add_trace(go.Bar(
    x=history["Date"],
    y=history["Volume"],
    marker=dict(
        color=["#00c853" if history["Close"].iloc[i] >= history["Open"].iloc[i]
               else "#ff4b4b" for i in range(len(history))],
        opacity=0.8,
        line=dict(color="rgba(0,0,0,0.1)", width=0.5)
    ),
    hovertemplate="<b>%{x}</b><br>Volume: %{y:,.0f}<extra></extra>",
    name="Volume"
))

fig_vol.update_layout(
    title=dict(
        text=f"<b>{TICKER} Daily Trading Volume — Last 30 Days</b>",
        font=dict(size=20, color="white"),
        x=0.5,
        xanchor="center"
    ),
    xaxis=dict(
        title=dict(text="Date", font=dict(color="white")),
        tickfont=dict(color="white"),
        showgrid=False,
        zeroline=False
    ),
    yaxis=dict(
        title=dict(text="Volume", font=dict(color="white")),
        tickfont=dict(color="white"),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#1a1a2e",
    height=380,
    margin=dict(t=80, b=60, l=60, r=40),
    showlegend=False
)

fig_vol.show()
fig_vol.write_image("iframe_figures/volume_chart.png", width=1000, height=500, scale=2)


## Step 6 Correlation Analysis

This is the most important part of the project.

We now ask the key question:

> **Does the sentiment of the news actually match what the stock price is doing?**

We will compare:
- The **sentiment score** we calculated from today's headlines
- The **price change** from yesterday's close to today's price

If sentiment is positive and the price is up → the signal is **confirmed** ✅  
If sentiment is positive but the price is down → the signal is **contradicted** ⚠️

This is exactly what hedge funds and quant traders analyze at a much larger scale.

In [ ]:
# Correlation Analysis

# Get current price data
price_data = get_stock_price(TICKER)
current_price = price_data["current_price"]
previous_close = price_data["previous_close"]

# Calculate price change
price_change = current_price - previous_close
price_change_pct = (price_change / previous_close) * 100

# Get sentiment score
sentiment_score = sentiment_results["average_score"]
sentiment_label = sentiment_results["overall_label"]

# Determine if sentiment and price agree
if sentiment_score > 0.1 and price_change > 0:
    signal = "✅ CONFIRMED"
    signal_color = "#00c853"
    signal_explanation = "Both sentiment and price are positive today. The news is aligned with market movement."
elif sentiment_score < -0.1 and price_change < 0:
    signal = "✅ CONFIRMED"
    signal_color = "#00c853"
    signal_explanation = "Both sentiment and price are negative today. The news is aligned with market movement."
elif sentiment_score > 0.1 and price_change < 0:
    signal = "⚠️ CONTRADICTED"
    signal_color = "#ffd700"
    signal_explanation = "Sentiment is positive but price dropped. The market may be reacting to other factors."
elif sentiment_score < -0.1 and price_change > 0:
    signal = "⚠️ CONTRADICTED"
    signal_color = "#ffd700"
    signal_explanation = "Sentiment is negative but price rose. The market may be ignoring the bad news for now."
else:
    signal = "➖ NEUTRAL"
    signal_color = "#ffd700"
    signal_explanation = "Sentiment is neutral. No strong signal either way."

# Print summary
print(f"{'='*55}")
print(f"  CORRELATION SUMMARY FOR {TICKER}")
print(f"{'='*55}")
print(f"  Sentiment Score   : {sentiment_score:+.4f} ({sentiment_label.upper()})")
print(f"  Price Yesterday   : ${previous_close:.2f}")
print(f"  Price Today       : ${current_price:.2f}")
print(f"  Price Change      : ${price_change:+.2f} ({price_change_pct:+.2f}%)")
print(f"{'='*55}")
print(f"  Signal            : {signal}")
print(f"  Explanation       : {signal_explanation}")
print(f"{'='*55}")

  CORRELATION SUMMARY FOR NVDA
  Sentiment Score   : +0.1020 (POSITIVE)
  Price Yesterday   : $213.66
  Price Today       : $204.56
  Price Change      : $-9.10 (-4.26%)
  Signal            : ⚠️ CONTRADICTED
  Explanation       : Sentiment is positive but price dropped. The market may be reacting to other factors.


In [30]:
# Correlation Visual

fig_corr = go.Figure()

# Sentiment bar
fig_corr.add_trace(go.Bar(
    x=["Sentiment Score"],
    y=[sentiment_score],
    marker=dict(
        color="#00c853" if sentiment_score > 0.1 else "#ff4b4b" if sentiment_score < -0.1 else "#ffd700",
        opacity=0.9,
        line=dict(color="rgba(0,0,0,0.1)", width=1)
    ),
    text=[f"{sentiment_score:+.2f}"],
    textposition="outside",
    textfont=dict(size=16, color="white"),
    name="Sentiment Score",
    width=0.3
))

# Price change bar (normalized to -1 to 1 scale for comparison)
max_change = 5  # assume 5% is a big move
normalized_price_change = max(-1, min(1, price_change_pct / max_change))

fig_corr.add_trace(go.Bar(
    x=["Price Change %"],
    y=[normalized_price_change],
    marker=dict(
        color="#00c853" if price_change_pct > 0 else "#ff4b4b",
        opacity=0.9,
        line=dict(color="rgba(0,0,0,0.1)", width=1)
    ),
    text=[f"{price_change_pct:+.2f}%"],
    textposition="outside",
    textfont=dict(size=16, color="white"),
    name="Price Change",
    width=0.3
))

fig_corr.add_hline(
    y=0,
    line_dash="dash",
    line_color="rgba(255,255,255,0.4)",
    line_width=1.5
)

fig_corr.update_layout(
    title=dict(
        text=f"<b>Sentiment vs Price Movement — {TICKER}</b>",
        font=dict(size=20, color="white"),
        x=0.5,
        xanchor="center"
    ),
    xaxis=dict(
        tickfont=dict(color="white", size=15),
        showgrid=False,
        zeroline=False
    ),
    yaxis=dict(
        title=dict(text="Normalized Score (-1 to +1)", font=dict(color="white")),
        range=[-1.4, 1.4],
        tickfont=dict(color="white"),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#1a1a2e",
    height=450,
    margin=dict(t=80, b=60, l=60, r=40),
    legend=dict(
        font=dict(color="white", size=13),
        bgcolor="rgba(0,0,0,0)"
    ),
    bargap=0.5
)

fig_corr.show()
fig_corr.write_image("iframe_figures/sentiment_vs_price.png", width=900, height=600, scale=2)

In [21]:
# Auto-generated Conclusion based on results

print(f"""
{'='*55}
  PROJECT SUMMARY
{'='*55}

  Stock Analyzed     : {TICKER} ({price_data['company_name']})
  Analysis Date      : {pd.Timestamp.now().strftime("%B %d, %Y")}

  SENTIMENT RESULTS
  -----------------
  Headlines Analyzed : {len(headlines)}
  Sentiment Score    : {sentiment_score:+.4f}
  Sentiment Label    : {sentiment_label.upper()}

  PRICE RESULTS
  -------------
  Previous Close     : ${previous_close:.2f}
  Current Price      : ${current_price:.2f}
  Price Change       : ${price_change:+.2f} ({price_change_pct:+.2f}%)

  SIGNAL
  ------
  {signal}
  {signal_explanation}

{'='*55}
""")


  PROJECT SUMMARY

  Stock Analyzed     : NVDA (NVIDIA Corporation)
  Analysis Date      : October 30, 2026

  SENTIMENT RESULTS
  -----------------
  Headlines Analyzed : 20
  Sentiment Score    : +0.1020
  Sentiment Label    : POSITIVE

  PRICE RESULTS
  -------------
  Previous Close     : $213.66
  Current Price      : $204.56
  Price Change       : $-9.10 (-4.26%)

  SIGNAL
  ------
  ⚠️ CONTRADICTED
  Sentiment is positive but price dropped. The market may be reacting to other factors.




## Limitations & Future Improvements

This project is a working proof of concept. Here are honest limitations and ideas to make it stronger:

### ⚠️ Limitations

- **One day snapshot:** sentiment is analyzed at a single point in time. A stronger analysis would track sentiment daily over weeks or months and look for patterns.

- **News sources only:** we are using NewsAPI which covers professional news. Adding Reddit and Twitter/X data would give a more complete picture of public sentiment.

- **Correlation ≠ Causation:** even when sentiment and price move in the same direction, it doesn't mean one caused the other. Many factors move stock prices simultaneously.

- **Free API limits:** NewsAPI's free tier limits us to 20 headlines per request and only recent articles. A paid tier or additional sources would improve coverage.

### Future Improvements

- **Track sentiment daily** and build a historical sentiment vs price dataset over time
- **Add Reddit scraping** using the `praw` library for social sentiment
- **Add more tickers** compare sentiment across multiple stocks simultaneously  
- **Train a custom model** fine-tuned on more recent financial data
- **Add price prediction** using sentiment as a feature in an ML model
- **Deploy the Streamlit dashboard** publicly on Streamlit Cloud

## Final Thoughts

This project demonstrates a complete **end-to-end data science pipeline**:

| Step | What We Did |
|---|---|
| Data Collection | Automatically scraped real financial headlines using NewsAPI |
| NLP & AI | Used FinBERT to score sentiment on financial text |
| Financial Data | Fetched live stock prices from Yahoo Finance |
| Visualization | Built interactive charts with Plotly |
| Analysis | Correlated sentiment signals with price movements |


Sentiment analysis captures the **votes**. Over time, the data does the weighing.
